# Laya fine-tune on Colab (GPU)

Use this when training on the M5 is too slow. Locally run `scripts/pack_colab.sh`, upload `dist/laya-colab.zip` to **MyDrive/laya-fine-tune/**, then run all cells.
Runtime → Change runtime type → **A100** (or L4). The base checkpoint `convaiinnovations/laya-multilingual` is public, so no HF token is needed.

Only the pre-tokenized training items and the training code are uploaded (token ids decode back to text, so treat the zip like the dataset: CC-BY-NC, research only).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/laya-fine-tune'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!rm -rf /content/laya-colab && unzip -q $DRIVE/laya-colab.zip -d /content/
%cd /content/laya-colab
!pip -q install -r requirements-train.txt
!ls -la data/build

## Train
`RUN=R1` uses all items; `RUN=R2` uses the NC-free items (order-confirm only). On CUDA, `train.py` picks bf16/fp16 autocast and turns gradient checkpointing off (1.25× faster).

In [ ]:
RUN = 'R1'   # or 'R2'
SUFFIX = '' if RUN == 'R1' else '_ncfree'
!python scripts/train.py --out checkpoints/{RUN} --items-suffix "{SUFFIX}" --device cuda

## Save to Drive
Download `checkpoints/<RUN>` into `models/laya-fine-tune/checkpoints/<RUN>` locally, then run `scripts/07_convert_mlx.py checkpoints/<RUN>` and `scripts/08_benchmark.py`.

In [ ]:
!mkdir -p $DRIVE/checkpoints && cp -r checkpoints/{RUN} $DRIVE/checkpoints/
!cat reports/runs.jsonl
!du -sh $DRIVE/checkpoints/{RUN}